# Rules and Weights Explanation Study Tutorial

This beginner-friendly notebook helps you compare rule-based and weight-based explanations. It creates balanced participant groups, builds question schedules, records the needed response fields, and provides a starting analysis.

The workflow runs the complete virtual experiment without training a model. Model and explanation generation are optional.

## Study design in plain language

- **Independent variable (IV):** something deliberately changed by the researcher. Here, the IVs are explanation display, explanation complexity, and whether the explanation is visible.
- **Dependent variable (DV):** the measured outcome. DV 1 is correct AI-label prediction; DV 2 is successfully changing one feature to flip the AI prediction.
- **Control variable (CV):** something held constant for a fair comparison. Controls include six displayed features, neutral Type 1/Type 2 labels, balanced AI labels, instructions, and feedback rules.
- **Between-participant factor:** different people receive different datasets, explanation displays, and complexity levels.
- **Within-participant factor:** the same person answers questions with and without the explanation.
- **Randomization:** chance determines group or question assignment, reducing bias.
- **Counterbalancing:** different condition orders are deliberately balanced. The task order here is fixed, so it is recorded and treated as a possible order limitation rather than counterbalanced.

Each participant completes 40 prediction questions followed by 40 change-the-prediction questions.

In [1]:
from pathlib import Path
import inspect
import os
import sys
import numpy as np
import pandas as pd

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'api.py').exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.api import xaikitTest
required_trial_args = {'num_training', 'num_testing'}
if not required_trial_args <= set(inspect.signature(xaikitTest.generate_trials).parameters):
    raise RuntimeError('This kernel has an older xaikitTest loaded. Restart the kernel, then Run All.')
from src.workflow_standard import PREDICTION_ONLY_METHOD
from src.xai_adapter import generate_surrogate_xai_methods

SEED = 42
OUTPUT_DIR = Path(os.environ.get('XAIKIT_TUTORIAL_OUTPUT_DIR', repo_root / 'tutorials' / 'rules_weights_output'))

## 1. Enter the study variables

`tested_w_xai` is balanced so each person receives the same number of questions with and without an explanation. The two tasks use a fixed order, so task is stored as a study phase rather than randomized.

The example uses 20 assignments per group only to exercise the planner. Choose the real sample size before recruitment using a power analysis or expert advice.

In [ ]:
study = xaikitTest(project_name='rules_weights_user_study', output_dir=OUTPUT_DIR)
study.add_iv('dataset', 'between', ['wine_quality', 'mushrooms'])
study.add_iv('xai_type', 'between', ['decision_tree', 'logistic_regression', 'hybrid'])
study.add_iv('complexity', 'between', ['low', 'high'])
study.add_iv('tested_w_xai', 'within', [False, True], randomization='trial')
study.add_cv('user_task', ['forward_simulation', 'counterfactual_simulation'])
study.add_cv('num_attributes_shown', [6])
study.add_cv('ai_label_balance', ['50_50'])
study.add_cv('xai_faithfulness', [0.9])
study.add_dv('forward_accuracy', ['continuous'])
study.add_dv('counterfactual_accuracy', ['continuous'])
pd.DataFrame({'role': ['IV', 'IV', 'IV', 'IV', 'DV', 'DV'], 'variable': ['dataset', 'xai_type', 'complexity', 'tested_w_xai', 'forward_accuracy', 'counterfactual_accuracy']})

## Complete the study setup

Define the research questions and participant-facing procedure before creating the walkthrough. The Python values make the setup reproducible; the interactive form lets a first-time researcher revise the same fields without writing more code. Saving an edit clears any earlier walkthrough approval.


In [ ]:
study.set_study_protocol(
    study_title='Comparing rule and weight displays',
    research_questions=[
        'RQ1: Does display type change prediction accuracy?',
        'RQ2: Does display type change success when editing one value to change the AI answer?',
        'RQ3: Do visibility and display complexity change these effects?',
    ],
    study_summary='Participants predict AI answers and edit one value to try to change an answer.',
    consent_text=('Replace this text with your approved consent information. Explain the purpose, '
                  'duration, voluntary participation, risks, data handling, compensation, and contacts.'),
    start_survey_questions=['What is your age range?', 'How familiar are you with AI systems?'],
    end_survey_questions=['How difficult was each task?', 'How helpful was the display?'],
    procedure_steps=[
        {'title': 'Welcome and consent', 'kind': 'consent', 'description': 'Read the consent page and choose whether to continue.', 'duration_min': 3},
        {'title': 'Start-of-study survey', 'kind': 'survey', 'description': 'Answer the planned background questions.', 'duration_min': 3},
        {'title': 'Instructions and practice', 'kind': 'practice', 'description': 'Learn both tasks and complete practice questions with feedback.', 'duration_min': 8},
        {'title': 'Prediction and editing questions', 'kind': 'trials', 'description': 'Complete 40 prediction questions followed by 40 editing questions.', 'duration_min': 25},
        {'title': 'End-of-study survey', 'kind': 'survey', 'description': 'Complete the planned ratings.', 'duration_min': 3},
        {'title': 'Debrief', 'kind': 'debrief', 'description': 'Show the debrief and researcher contact information.', 'duration_min': 1},
    ],
)
protocol_editor = study.edit_study_protocol()

## 2. Load example data and make a balanced question template

This worked example uses six Wine Quality features. Repeat the preparation steps for every dataset used in the real study. The cell creates 40 questions per assignment, with 20 explanation-visible and 20 explanation-hidden questions.

In [ ]:
data = study.prepare_dataset(
    'wine_quality', feature_cols=['Alcohol', 'Sulphates', 'SO2', 'Vinegar Taint', 'pH', 'residual sugar'],
    rank_features_by_target=False, model_type='mlp', test_size=0.5, random_state=SEED,
)
base = study.generate_trials(
    participants_per_between_condition=20, num_training=10, num_testing=40,
    shuffle_instances=True,
    output_dir='base_trials', seed=SEED, preview_rows=4,
)
all_base_trials = pd.DataFrame(base.trials)
training_trials = all_base_trials.query('phase == "training"').copy()
base_trials = all_base_trials.query('phase == "testing"').copy()
assert (all_base_trials.groupby('participantId').head(10)['phase'] == 'training').all()
assert training_trials['tested_w_xai'].isna().all()
assert (base_trials.groupby('participantId')['tested_w_xai'].value_counts() == 20).all()
base_trials.groupby(['dataset', 'xai_type', 'complexity'])['participantId'].nunique().head(12)

## 3. Build the two task phases

The next cell creates a 40-question prediction phase and a 40-question change-the-prediction phase. Both phases remain balanced between visible and hidden explanations. In a live study, use a different randomized case order for the second phase so participants do not simply remember earlier answers.

In [ ]:
forward = base_trials.copy()
forward['task'] = 'forward_simulation'
forward['task_order'] = 1
forward['task_trial'] = forward.groupby('participantId').cumcount() + 1

counterfactual = base_trials.copy()
counterfactual['task'] = 'counterfactual_simulation'
counterfactual['task_order'] = 2
counterfactual['task_trial'] = counterfactual.groupby('participantId').cumcount() + 1

schedule = pd.concat([forward, counterfactual], ignore_index=True).sort_values(['participantId', 'task_order', 'task_trial'])
assert (schedule.groupby(['participantId', 'task', 'tested_w_xai']).size() == 20).all()
schedule.head()

## 4. Optional: train the AI and create explanation variants

A low-complexity rule display uses a shallower tree; a low-complexity weight display shows only the strongest weights. High-complexity displays show more detail. The hybrid group sees either rules or weights on explanation-visible questions.

Leave `RUN_MODEL_PIPELINE=False` while learning the planning workflow. Change it to `True` only when the required model packages are installed.

In [ ]:
RUN_MODEL_PIPELINE = False
if RUN_MODEL_PIPELINE:
    study.train_AI_model(model_type='mlp', batch_size=64, train_kwargs={'epochs': 50}, verbose=True)
    ai_predictions = study.trained_ai_model.predict(data.X_test)
    surrogate_df = pd.DataFrame(data.X_test, columns=data.raw_feature_names)
    surrogate_df['prediction'] = np.asarray(ai_predictions).reshape(-1)
    generated = generate_surrogate_xai_methods(
        dataframe=surrogate_df, instance_ids=data.test_instance_ids,
        app_id='wine_quality', model_name='mlp',
        methods=('decision_tree', 'logistic_regression'), depths=(2, 3),
        variants=('sparse', 'dense'), top_k=3, random_state=SEED,
    )
    print(generated)
else:
    print('Planning cells are complete. Set RUN_MODEL_PIPELINE=True for model and explanation generation.')

## 5. Check the question set before launch

A fair comparison needs similar question difficulty across groups. Check that Type 1 and Type 2 labels are evenly represented and that each explanation describes the AI with the intended level of agreement. These are properties of the prepared question set, not participant outcomes.

For the hybrid group, the cell also randomly chooses whether a visible explanation uses rules or weights. Save this assignment in the trial file.

In [ ]:
def audit_stimulus_controls(stimuli):
    by_session = stimuli.groupby(['participantId', 'task'])
    label_share = by_session['ai_label'].mean()
    fidelity = by_session.apply(lambda g: np.mean(g['ai_label'].to_numpy() == g['xai_label'].to_numpy()), include_groups=False)
    return pd.DataFrame({'positive_label_share': label_share, 'xai_ai_agreement': fidelity})

rng = np.random.default_rng(SEED)
hybrid_mask = (schedule['xai_type'] == 'hybrid') & schedule['tested_w_xai'].astype(bool)
schedule.loc[hybrid_mask, 'shown_xai_type'] = rng.choice(['decision_tree', 'logistic_regression'], size=hybrid_mask.sum())
schedule.loc[(schedule['xai_type'] != 'hybrid') & schedule['tested_w_xai'].astype(bool), 'shown_xai_type'] = schedule['xai_type']
schedule.loc[~schedule['tested_w_xai'].astype(bool), 'shown_xai_type'] = 'none'
schedule[['xai_type', 'tested_w_xai', 'shown_xai_type']].value_counts()

## 6. Decide what to record

For prediction questions, record the selected Type 1/Type 2 label and whether it matches the AI. For change-the-prediction questions, record the selected feature, original value, edited value, new AI label, and whether the label changed. Record response time for both tasks.

Keep participant identifiers anonymous, store consent information separately, and decide exclusion rules before data collection.

In [ ]:
response_schema = pd.DataFrame(columns=[
    'participantId', 'dataset', 'xai_type', 'complexity', 'task', 'task_trial',
    'tested_w_xai', 'shown_xai_type', 'instanceId', 'human_label', 'ai_label',
    'forward_correct', 'selected_attribute', 'original_value', 'edited_value',
    'edited_ai_label', 'counterfactual_correct', 'response_time_s', 'comprehension_pass',
])
response_schema

## 7. Compare accuracy across conditions

Analyze the two tasks separately because they measure different skills. The model below checks explanation display, visibility, complexity, and question number. It also groups repeated answers from the same participant so they are not treated as independent people.

Treat this as a starting template. Have the final analysis reviewed and fixed before recruitment.

In [ ]:
import statsmodels.formula.api as smf

def fit_task_accuracy_model(human_responses, task, outcome):
    df = human_responses.query('task == @task and comprehension_pass == True').copy()
    df['tested_w_xai'] = df['tested_w_xai'].astype(int)
    formula = f'{outcome} ~ task_trial + C(xai_type) * tested_w_xai + C(xai_type):C(complexity)'
    return smf.mixedlm(formula, data=df, groups=df['participantId']).fit(reml=False)

print('Prediction task: fit_task_accuracy_model(responses, "forward_simulation", "forward_correct")')
print('Change task: fit_task_accuracy_model(responses, "counterfactual_simulation", "counterfactual_correct")')

## 8. Replication checks and expected result pattern

The reference validation crosses two datasets, three explanation groups, and two complexity levels between participants. Each person completes 40 prediction questions and then 40 change-the-prediction questions, balanced between visible and hidden explanations. Low complexity uses tree depth 2 or three non-zero weights; high complexity uses tree depth 3 or six non-zero weights. AI labels are balanced 50/50, and the displayed explanation agrees with the AI on 90% of selected cases.

Expected descriptive patterns are listed below. Analyze datasets and tasks separately, and report deviations honestly.

In [ ]:
expected_patterns = pd.DataFrame([
    {'check': 'task difficulty', 'expected_pattern': 'change-the-prediction accuracy is lower than prediction accuracy'},
    {'check': 'rule visibility', 'expected_pattern': 'Rules improve when visible because deeper paths are difficult to recall'},
    {'check': 'weight recall', 'expected_pattern': 'Weights are more likely than Rules to remain useful when hidden'},
    {'check': 'dataset interaction', 'expected_pattern': 'Rules and Weights perform differently across datasets'},
    {'check': 'change task', 'expected_pattern': 'Weights often support more accurate changes than Rules, especially for numeric data'},
])
expected_patterns

## 9. Run the complete study with virtual participants

This cell executes both task phases with the KNN baseline from the cognitive-model API. The executor fits a fresh KNN on each participant-condition training phase, then records its responses on held-out testing questions. Its output is machine-proxy data, not evidence about people.

The `prediction_pool` supplies the AI answer for every case. The summaries keep prediction and change-the-prediction accuracy separate because they are different DVs.


In [ ]:
schedule = schedule.copy()
schedule['phase'] = 'testing'
all_trials = pd.concat([training_trials, schedule], ignore_index=True, sort=False)
all_trials['trialId'] = all_trials.groupby('participantId').cumcount() + 1
all_trials['phaseTrialId'] = all_trials.groupby(['participantId', 'phase']).cumcount() + 1
all_trials['dataId'] = data.dataset_id
all_trials['xai_method'] = all_trials['xai_type']
study.trials = all_trials.to_dict('records')
study.trained_ai_model = object()  # Predictions are already supplied below.
study.model_name = 'tutorial_ai'
trial_ids = sorted(all_trials['instanceId'].astype(int).unique())
predictions = np.asarray([int(data.split.y[i]) for i in trial_ids])
surrogate_training = pd.DataFrame(data.split.X_model, columns=data.model_feature_names)
surrogate_training['instanceId'] = np.arange(len(surrogate_training))
surrogate_training['pred'] = np.asarray(data.split.y)
generated_proxy_xai = generate_surrogate_xai_methods(
    dataframe=surrogate_training, instance_ids=np.arange(len(surrogate_training)),
    app_id=data.dataset_id, model_name=study.model_name,
    methods=('decision_tree', 'logistic_regression'), depths=(3,),
    variants=('sparse',), variant='sparse', top_k=3, random_state=SEED,
    feature_columns=data.model_feature_names,
)
X_trials = data.split.X_model[trial_ids]
rule_frame = generated_proxy_xai.methods['rules'].explain(X_trials).to_explanation_df(
    trial_ids, predictions, data.dataset_id, study.model_name,
)
rule_frame['expMethod'] = 'decision_tree'
weight_frame = generated_proxy_xai.methods['weights'].explain(X_trials).to_explanation_df(
    trial_ids, predictions, data.dataset_id, study.model_name,
)
weight_frame['expMethod'] = 'logistic_regression'
hybrid_frame = rule_frame.copy()
attribution_columns = [column for column in rule_frame if column.startswith('a') and column.endswith('_i')]
hybrid_frame[attribution_columns] = (
    rule_frame[attribution_columns].to_numpy() + weight_frame[attribution_columns].to_numpy()
) / 2.0
hybrid_frame['expMethod'] = 'hybrid'
prediction_pool = pd.concat([rule_frame, weight_frame, hybrid_frame], ignore_index=True)
study.set_cognitive_model(
    cognitive_model_id='knn',
    model_kwargs={'n_neighbors': 3},
)

## 10. Preview and approve the complete participant journey

Use **Back** and **Next** to inspect consent, the start survey, practice instructions, and all 80 questions for one participant in their actual order. Training always appears first. During testing, the randomized `tested_w_xai` value selects the feature-plus-surrogate-vector KNN or the feature-only KNN. The walkthrough ends with the post-task survey and debrief. Tick the confirmation and click **Approve walkthrough** on the last page.


In [ ]:
walkthrough_pages = study.preview_experiment_walkthrough(
    participant_id=1, explanation_pool=prediction_pool, visualization='importance',
)

## 11. Execute only after approval

Execution is locked until the walkthrough has been approved. If the setup form is saved again, return to the walkthrough and approve the revised version.


In [ ]:
protocol_path = study.save_study_protocol()
simulated_results = study.run_experiment(
    mode='whole_experiment', participant_id=None, explanation_pool=prediction_pool,
    require_walkthrough_approval=True,
)
csv_path, json_path = study.save_results(out_dir='simulated_results')
forward_summary = simulated_results.query('task == \"forward_simulation\"').groupby(
    ['dataset', 'xai_type', 'complexity', 'tested_w_xai'], as_index=False
).agg(accuracy=('forward_accuracy', 'mean'), questions=('trialId', 'size'))
counterfactual_summary = simulated_results.query('task == \"counterfactual_simulation\"').groupby(
    ['dataset', 'xai_type', 'complexity', 'tested_w_xai'], as_index=False
).agg(accuracy=('counterfactual_accuracy', 'mean'), questions=('trialId', 'size'))
print(f'Saved the setup to {protocol_path}')
print(f'Saved {len(simulated_results):,} virtual responses to {csv_path} and {json_path}')
iv_dv_analysis = study.analyze_iv_dv(iv='xai_type', dv='forward_accuracy')
display(forward_summary, counterfactual_summary, iv_dv_analysis.descriptives)

## Before recruiting anyone

Freeze the stimuli, exclusion rules, sample-size rationale, and analysis plan. Obtain ethics approval where required. Keep machine-proxy files separate from human responses, and replace the tutorial prediction table with finalized model artifacts for a full replication.